# Panel HMM for 10 Global Equity Markets — Price + VIX, 5-Regime Version

This notebook rebuilds the original full-sample Panel HMM specification, but changes the model from **4 regimes** to **5 regimes**.

Final feature set:

```python
FEATURES_PRICE_VIX = [
    "mom_21", "mom_63",
    "vol_21", "vol_63",
    "drawdown",
    "vix_level", "vix_chg_21"
]
```

Design:

- China CSI 300 is loaded from `CSI300.xlsx`.
- Other 9 markets are downloaded from `yfinance`.
- The model uses the strict common-date price dataframe for all 10 markets.
- FX is excluded.
- `ret_1d` is retained for labelling and validation, but is not used as an HMM feature.
- HMM regimes are pooled/common across all markets.
- Final output includes **two different ranking methods**:
  1. **Method 1: Boom minus Crisis**
  2. **Method 2: Good-regime basket minus Bad-regime basket**


In [ ]:
# ============================================================
# Cell 1: Imports and configuration
# ============================================================

# !pip install yfinance hmmlearn openpyxl

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

from sklearn.preprocessing import RobustScaler
from hmmlearn.hmm import GaussianHMM

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

START_DATE = "2000-01-01"      # original full-sample request
END_DATE = None                 # keep None to fetch latest available
CHINA_XLSX = "CSI300.xlsx"      # keep in same folder as notebook/script

N_REGIMES = 5
REGIME_NAMES = {
    0: "Crisis",
    1: "Correction",
    2: "Recovery",
    3: "Growth",
    4: "Boom",
}
ORDERED_REGIME_LIST = [REGIME_NAMES[i] for i in range(N_REGIMES)]

TRADING_DAYS_1Y = 252
TRADING_DAYS_2Y = 504

FEATURES_PRICE_VIX = [
    "mom_21", "mom_63",
    "vol_21", "vol_63",
    "drawdown",
    "vix_level", "vix_chg_21"
]

print("5-regime Panel HMM")
print("Features used:")
print(FEATURES_PRICE_VIX)


In [ ]:
# ============================================================
# Cell 2: Market tickers and yfinance cleaning helper
# ============================================================

YF_MARKETS = {
    "USA_SP500": ["^GSPC"],
    "UK_FTSE100": ["^FTSE"],
    "Japan_Nikkei225": ["^N225"],
    "Taiwan_TAIEX": ["^TWII"],
    "Korea_KOSPI": ["^KS11"],
    "Indonesia_JakartaComposite": ["^JKSE"],
    "India_Nifty100": ["^CNX100"],
    "Brazil_Ibovespa": ["^BVSP"],
    "SouthAfrica_JSETop40": ["^J200.JO", "J20U.L", "JN0U.FGI"],
}

ORDER = [
    "USA_SP500",
    "UK_FTSE100",
    "China_CSI300",
    "Japan_Nikkei225",
    "Taiwan_TAIEX",
    "Korea_KOSPI",
    "Indonesia_JakartaComposite",
    "India_Nifty100",
    "Brazil_Ibovespa",
    "SouthAfrica_JSETop40",
]


def _clean_yf_download(raw):
    # Extract a clean Close series from yfinance output.
    if raw is None or raw.empty:
        return None

    df = raw.copy()

    if isinstance(df.columns, pd.MultiIndex):
        level0 = df.columns.get_level_values(0)
        if "Close" in level0:
            close = df.xs("Close", level=0, axis=1).iloc[:, 0]
        elif "Adj Close" in level0:
            close = df.xs("Adj Close", level=0, axis=1).iloc[:, 0]
        else:
            return None
    else:
        if "Close" in df.columns:
            close = df["Close"]
        elif "Adj Close" in df.columns:
            close = df["Adj Close"]
        else:
            return None

    close = pd.to_numeric(close, errors="coerce").dropna()
    close.index = pd.to_datetime(close.index).tz_localize(None).normalize()
    close = close[~close.index.duplicated(keep="last")].sort_index()
    return close


def download_first_working_ticker(market_name, candidates):
    # Try candidate Yahoo tickers and return the first non-empty Close series.
    for ticker in candidates:
        try:
            raw = yf.download(
                ticker,
                start=START_DATE,
                end=END_DATE,
                interval="1d",
                auto_adjust=True,
                progress=False,
                threads=False,
            )
            close = _clean_yf_download(raw)
            if close is not None and not close.empty:
                close.name = market_name
                return close, ticker
        except Exception as err:
            print(f"Failed {market_name} / {ticker}: {err}")
    return None, None


In [ ]:
# ============================================================
# Cell 3: Load China CSI 300 from Excel
# ============================================================

def load_china_csi300(path=CHINA_XLSX):
    # Load China CSI 300 from Excel with Date and Close columns.
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"{path} not found. Keep CSI300.xlsx in the same folder as this notebook."
        )

    csi = pd.read_excel(path)
    csi.columns = [str(c).strip() for c in csi.columns]

    if "Date" not in csi.columns or "Close" not in csi.columns:
        raise ValueError("CSI300.xlsx must contain columns named Date and Close.")

    if pd.api.types.is_numeric_dtype(csi["Date"]):
        csi["Date"] = pd.to_datetime(csi["Date"], unit="D", origin="1899-12-30", errors="coerce")
    else:
        csi["Date"] = pd.to_datetime(csi["Date"], errors="coerce")

    csi["Close"] = pd.to_numeric(csi["Close"], errors="coerce")
    csi = csi.dropna(subset=["Date", "Close"])
    csi["Date"] = csi["Date"].dt.tz_localize(None).dt.normalize()
    csi = csi.sort_values("Date").drop_duplicates("Date", keep="last")
    csi = csi[csi["Date"] >= pd.Timestamp(START_DATE)]

    out = csi.set_index("Date")["Close"].rename("China_CSI300")
    return out


In [ ]:
# ============================================================
# Cell 4: Build 10-market price dataframe
# ============================================================

series_list = []
availability_rows = []

for market, candidates in YF_MARKETS.items():
    s, selected_ticker = download_first_working_ticker(market, candidates)

    if s is None or s.empty:
        availability_rows.append({
            "Market": market,
            "Source": "yfinance",
            "Ticker": ", ".join(candidates),
            "First_Date": pd.NaT,
            "Last_Date": pd.NaT,
            "Observations": 0,
            "Status": "No data fetched",
        })
        continue

    series_list.append(s)
    availability_rows.append({
        "Market": market,
        "Source": "yfinance",
        "Ticker": selected_ticker,
        "First_Date": s.index.min(),
        "Last_Date": s.index.max(),
        "Observations": int(s.shape[0]),
        "Status": "OK",
    })

china = load_china_csi300(CHINA_XLSX)
series_list.append(china)
availability_rows.append({
    "Market": "China_CSI300",
    "Source": "Uploaded Excel",
    "Ticker": "CSI300.xlsx",
    "First_Date": china.index.min(),
    "Last_Date": china.index.max(),
    "Observations": int(china.shape[0]),
    "Status": "OK",
})

availability = pd.DataFrame(availability_rows)
availability["First_Date"] = pd.to_datetime(availability["First_Date"])
availability["Last_Date"] = pd.to_datetime(availability["Last_Date"])
availability = availability.sort_values("Market").reset_index(drop=True)

market_prices_10 = pd.concat(series_list, axis=1).sort_index()
market_prices_10 = market_prices_10.loc[market_prices_10.index >= pd.Timestamp(START_DATE)]
market_prices_10.index.name = "Date"

missing_markets = sorted(set(ORDER) - set(market_prices_10.columns))
if missing_markets:
    raise ValueError(f"Missing markets in price dataframe: {missing_markets}")

market_prices_10 = market_prices_10[ORDER]

first_valid_dates = market_prices_10.apply(lambda x: x.first_valid_index())
common_start_candidate = first_valid_dates.max()
market_prices_10_common = (
    market_prices_10
    .loc[market_prices_10.index >= common_start_candidate]
    .dropna(how="any")
    .copy()
)

market_returns_10_common = market_prices_10_common.pct_change(fill_method=None).dropna(how="any")

common_diagnostics = pd.DataFrame({
    "Metric": [
        "Requested start date",
        "Theoretical common start date",
        "Actual first common trading date",
        "Outer price rows",
        "Strict common price rows",
        "Strict common return rows",
    ],
    "Value": [
        START_DATE,
        common_start_candidate.date(),
        market_prices_10_common.index.min().date(),
        market_prices_10.shape[0],
        market_prices_10_common.shape[0],
        market_returns_10_common.shape[0],
    ]
})

print("Availability:")
display(availability)
print("\nCommon diagnostics:")
display(common_diagnostics)
print("\nCommon price shape:", market_prices_10_common.shape)
display(market_prices_10_common.head())
display(market_prices_10_common.tail())


In [ ]:
# ============================================================
# Cell 5: Create panel features market-by-market
# ============================================================

def make_market_features(price_series, market_name):
    s = pd.Series(price_series).copy()
    s.index = pd.to_datetime(s.index)
    s = pd.to_numeric(s, errors="coerce").dropna().sort_index()

    df = pd.DataFrame(index=s.index)
    df["Date"] = s.index
    df["Market"] = market_name
    df["Close"] = s.values

    # Retained for regime labelling/validation, not used as HMM feature.
    df["ret_1d"] = s.pct_change()

    df["mom_21"] = s.pct_change(21)
    df["mom_63"] = s.pct_change(63)
    df["vol_21"] = df["ret_1d"].rolling(21).std() * np.sqrt(252)
    df["vol_63"] = df["ret_1d"].rolling(63).std() * np.sqrt(252)
    df["drawdown"] = s / s.cummax() - 1

    return df.reset_index(drop=True)

panel_list = []
for market in market_prices_10_common.columns:
    temp = make_market_features(market_prices_10_common[market], market)
    panel_list.append(temp)

panel_df_raw = pd.concat(panel_list, axis=0).reset_index(drop=True)
panel_df = panel_df_raw.dropna().reset_index(drop=True)

print("Before dropna:", panel_df_raw.shape)
print("After dropna: ", panel_df.shape)
print("\nObservations per market:")
print(panel_df["Market"].value_counts())
display(panel_df.head())


In [ ]:
# ============================================================
# Cell 6: Add VIX features
# ============================================================

vix_raw = yf.download(
    "^VIX",
    start=market_prices_10_common.index.min().strftime("%Y-%m-%d"),
    end=END_DATE,
    interval="1d",
    auto_adjust=True,
    progress=False,
    threads=False,
)

vix = _clean_yf_download(vix_raw)
if vix is None or vix.empty:
    raise ValueError("VIX data could not be downloaded from yfinance.")

vix.name = "VIX"
vix_df = pd.DataFrame(index=vix.index)
vix_df["Date"] = vix_df.index
vix_df["vix_level"] = vix
vix_df["vix_chg_21"] = vix.pct_change(21)
vix_df["vix_z_252"] = (vix - vix.rolling(252).mean()) / vix.rolling(252).std()
vix_df = vix_df.reset_index(drop=True)

for c in ["vix_level", "vix_chg_21", "vix_z_252"]:
    if c in panel_df.columns:
        panel_df = panel_df.drop(columns=[c])

panel_df = panel_df.merge(vix_df, on="Date", how="left")

print("Panel after VIX merge:", panel_df.shape)
print("\nMissing values in selected HMM features:")
print(panel_df[FEATURES_PRICE_VIX].isna().sum())


In [ ]:
# ============================================================
# Cell 7: Final cleaning for 5-regime Price + VIX feature set
# ============================================================

FEATURES = FEATURES_PRICE_VIX.copy()

panel_df = panel_df.sort_values(["Market", "Date"]).reset_index(drop=True)
panel_clean = panel_df.dropna(subset=FEATURES + ["ret_1d"]).reset_index(drop=True)

print("Rows before feature dropna:", len(panel_df))
print("Rows after feature dropna: ", len(panel_clean))
print("\nObservations per market after cleaning:")
print(panel_clean["Market"].value_counts())
print("\nFeature columns used in HMM:")
print(FEATURES)


## Pooled HMM setup

The HMM is fit on one pooled panel. Each market is robust-scaled using its own history before pooling, so regime features are interpreted relative to each market's own normal behavior.


In [ ]:
# ============================================================
# Cell 8: Per-market RobustScaler standardization
# ============================================================

scaled_parts = []
market_scalers = {}

# Preserve raw variables for interpretation before scaling.
raw_keep_cols = ["ret_1d"] + FEATURES

for market, grp in panel_clean.groupby("Market", sort=False):
    grp = grp.copy()

    for col in raw_keep_cols:
        grp[f"{col}_raw"] = grp[col]

    scaler = RobustScaler()
    grp[FEATURES] = scaler.fit_transform(grp[FEATURES])
    market_scalers[market] = scaler
    scaled_parts.append(grp)

panel_scaled = (
    pd.concat(scaled_parts, axis=0)
      .sort_values(["Market", "Date"])
      .reset_index(drop=True)
)

X = panel_scaled[FEATURES].astype(float).values
lengths = panel_scaled.groupby("Market", sort=False).size().values

assert lengths.sum() == len(X), "Sequence lengths must sum to total rows."
assert np.isnan(X).sum() == 0, "NaN found in X."
assert np.isinf(X).sum() == 0, "Inf found in X."

print("Scaled panel shape:", panel_scaled.shape)
print("X shape:", X.shape)
print("Lengths:", lengths)
print("\nSanity check after RobustScaler:")
display(panel_scaled[FEATURES].agg(["mean", "std"]).round(4))


In [ ]:
# ============================================================
# Cell 9: Fit pooled 5-regime Gaussian HMM
# ============================================================

RANDOM_STATE = 42

panel_hmm = GaussianHMM(
    n_components=N_REGIMES,
    covariance_type="full",
    n_iter=1000,
    tol=1e-4,
    random_state=RANDOM_STATE,
    verbose=False,
    min_covar=1e-4,
)

panel_hmm.fit(X, lengths=lengths)

print("Converged:", panel_hmm.monitor_.converged)
print("Iterations run:", panel_hmm.monitor_.iter)
print("Log-likelihood:", panel_hmm.score(X, lengths=lengths))

panel_scaled["regime_raw"] = panel_hmm.predict(X, lengths=lengths)

transition_shared_raw = pd.DataFrame(
    panel_hmm.transmat_,
    index=[f"Raw_State_{i}" for i in range(N_REGIMES)],
    columns=[f"Raw_State_{i}" for i in range(N_REGIMES)],
)

display(transition_shared_raw.round(4))
print("\nRaw state counts:")
print(panel_scaled["regime_raw"].value_counts().sort_index())


In [ ]:
# ============================================================
# Cell 10: Relabel 5 regimes by raw mean daily return
# ============================================================

regime_order = (
    panel_scaled.groupby("regime_raw")["ret_1d_raw"]
    .mean()
    .sort_values()
    .index.tolist()
)

label_map = {old_raw_state: ordered_state for ordered_state, old_raw_state in enumerate(regime_order)}
panel_scaled["regime"] = panel_scaled["regime_raw"].map(label_map).astype(int)
panel_scaled["regime_label"] = panel_scaled["regime"].map(REGIME_NAMES)

print("Raw component -> ordered regime index:")
print(label_map)

regime_char = (
    panel_scaled
    .groupby("regime_label")
    .agg(
        N=("ret_1d_raw", "size"),
        Mean_Daily_Return=("ret_1d_raw", "mean"),
        Ann_Return_Proxy=("ret_1d_raw", lambda x: x.mean() * 252),
    )
    .reindex(ORDERED_REGIME_LIST)
)

display(regime_char.round(4))


In [ ]:
# ============================================================
# Cell 11: Full regime validation table
# ============================================================

regime_validation = (
    panel_scaled
    .groupby("regime_label")
    .agg(
        N=("ret_1d_raw", "size"),
        Mean_Daily_Return=("ret_1d_raw", "mean"),
        Median_Daily_Return=("ret_1d_raw", "median"),
        Ann_Return_Proxy=("ret_1d_raw", lambda x: x.mean() * 252),
        Avg_Mom_21=("mom_21_raw", "mean"),
        Avg_Mom_63=("mom_63_raw", "mean"),
        Avg_Vol_21=("vol_21_raw", "mean"),
        Avg_Vol_63=("vol_63_raw", "mean"),
        Avg_Drawdown=("drawdown_raw", "mean"),
        Avg_VIX=("vix_level_raw", "mean"),
        Avg_VIX_Change_21=("vix_chg_21_raw", "mean"),
    )
    .reindex(ORDERED_REGIME_LIST)
)

display(regime_validation.round(4))


In [ ]:
# ============================================================
# Cell 12: Regime share by market
# ============================================================

regime_share_by_market = (
    pd.crosstab(
        panel_scaled["Market"],
        panel_scaled["regime_label"],
        normalize="index"
    ) * 100
)

for col in ORDERED_REGIME_LIST:
    if col not in regime_share_by_market.columns:
        regime_share_by_market[col] = 0.0

regime_share_by_market = regime_share_by_market[ORDERED_REGIME_LIST]
display(regime_share_by_market.round(2))


In [ ]:
# ============================================================
# Cell 13: Current regime probability table
# ============================================================

state_probs_raw = panel_hmm.predict_proba(X, lengths=lengths)

for i in range(N_REGIMES):
    panel_scaled[f"Prob_Raw_{i}"] = state_probs_raw[:, i]

for raw_state, ordered_state in label_map.items():
    regime_name = REGIME_NAMES[ordered_state]
    panel_scaled[f"P_{regime_name}"] = panel_scaled[f"Prob_Raw_{raw_state}"]

current_rows = []
for market, grp in panel_scaled.groupby("Market", sort=False):
    latest = grp.sort_values("Date").iloc[-1]
    prob_values = {f"P_{name}": latest[f"P_{name}"] for name in ORDERED_REGIME_LIST}
    row = {
        "Market": market,
        "Latest_Date": latest["Date"],
        "Current_Regime": latest["regime_label"],
        **prob_values,
        "Max_Probability": max(prob_values.values()),
    }
    current_rows.append(row)

current_regime_prob_table = pd.DataFrame(current_rows).set_index("Market")
display(current_regime_prob_table.round(3))


In [ ]:
# ============================================================
# Cell 14: Per-market empirical transition matrices and 1Y/2Y projections
# ============================================================

def empirical_transition_matrix(regime_seq, n_states=N_REGIMES):
    # Row-normalized transition matrix from an ordered regime sequence.
    M = np.zeros((n_states, n_states), dtype=float)
    for t in range(len(regime_seq) - 1):
        M[regime_seq[t], regime_seq[t + 1]] += 1

    row_sums = M.sum(axis=1, keepdims=True)
    zero_rows = (row_sums[:, 0] == 0)
    row_sums[row_sums == 0] = 1
    T = M / row_sums
    if zero_rows.any():
        T[zero_rows, :] = 1.0 / n_states
    return T


def stationary_distribution(T, steps=5000):
    # Stable stationary approximation by repeated multiplication.
    p = np.full(T.shape[0], 1.0 / T.shape[0])
    for _ in range(steps):
        p = p @ T
    return p / p.sum()

transition_matrices = {}
projection_rows = []

for market, grp in panel_scaled.groupby("Market", sort=False):
    grp = grp.sort_values("Date")
    seq = grp["regime"].astype(int).values

    T = empirical_transition_matrix(seq, N_REGIMES)
    transition_matrices[market] = T

    current_state = seq[-1]
    current_label = REGIME_NAMES[current_state]

    e_current = np.zeros(N_REGIMES)
    e_current[current_state] = 1.0

    dist_1y = e_current @ np.linalg.matrix_power(T, TRADING_DAYS_1Y)
    dist_2y = e_current @ np.linalg.matrix_power(T, TRADING_DAYS_2Y)
    stat = stationary_distribution(T)

    row = {
        "Market": market,
        "Current_Regime": current_label,
        "Daily_Persistence": T[current_state, current_state],
    }

    for i, name in REGIME_NAMES.items():
        row[f"P_{name}_1Y"] = dist_1y[i]
    for i, name in REGIME_NAMES.items():
        row[f"P_{name}_2Y"] = dist_2y[i]
    for i, name in REGIME_NAMES.items():
        row[f"Stationary_Pct_{name}"] = stat[i]

    projection_rows.append(row)

projection_df = pd.DataFrame(projection_rows).set_index("Market")
display(projection_df.round(3))


## Two compatible ranking methods for a 5-regime HMM

Because this model has **5 regimes**, the ranking should not be forced into the old 4-regime comparison logic.

The regimes are first ordered from weakest to strongest:

```text
Crisis → Correction → Recovery → Growth → Boom
```

Then each regime is mapped to a normalized score between 0 and 1:

```text
Crisis = 0.00
Correction = 0.25
Recovery = 0.50
Growth = 0.75
Boom = 1.00
```

This makes the 5-regime ranking more comparable to other regime specifications.

### Method 1: Expected Normalized Regime Score

This uses the full 5-regime distribution:

```text
Method 1 = Σ projected probability of regime × normalized regime score
```

Higher score means the market is expected to spend more probability mass in stronger regimes.

### Method 2: Tail Balance Score

This uses the top and bottom regime groups:

```text
Method 2 = P(Growth or Boom) - P(Crisis or Correction)
```

Recovery is treated as neutral.

This is better than simply `Boom - Crisis` because the 5-regime model has a middle state and two positive/two negative-side states.


In [ ]:
# ============================================================
# Cell 15: Compatible Ranking Method 1 — Expected Normalized Regime Score
# ============================================================

ranking_method1 = projection_df.copy()

# Map ordered regimes to a common 0-to-1 scale.
# This makes the 5-regime model comparable with other regime counts.
REGIME_SCORE = {
    name: i / (N_REGIMES - 1)
    for i, name in enumerate(ORDERED_REGIME_LIST)
}

print("Normalized regime score map:")
print(REGIME_SCORE)

# Weighted 1Y/2Y projected probabilities for each regime.
for name in ORDERED_REGIME_LIST:
    ranking_method1[f"Weighted_{name}"] = (
        0.6 * ranking_method1[f"P_{name}_1Y"] +
        0.4 * ranking_method1[f"P_{name}_2Y"]
    )

# Expected normalized regime score: uses all 5 regimes.
ranking_method1["Method1_Expected_Regime_Score"] = 0.0

for name in ORDERED_REGIME_LIST:
    ranking_method1["Method1_Expected_Regime_Score"] += (
        REGIME_SCORE[name] * ranking_method1[f"Weighted_{name}"]
    )

# Optional interpretability columns
ranking_method1["Expected_Regime_Index_0_to_4"] = (
    ranking_method1["Method1_Expected_Regime_Score"] * (N_REGIMES - 1)
)

ranking_method1 = ranking_method1.sort_values(
    "Method1_Expected_Regime_Score",
    ascending=False
)

ranking_method1["Rank_Method1"] = np.arange(1, len(ranking_method1) + 1)

method1_cols = [
    "Rank_Method1",
    "Current_Regime",
    "Daily_Persistence",
    "Method1_Expected_Regime_Score",
    "Expected_Regime_Index_0_to_4",
    "Weighted_Crisis",
    "Weighted_Correction",
    "Weighted_Recovery",
    "Weighted_Growth",
    "Weighted_Boom",
]

print("Method 1: Expected normalized regime score")
display(ranking_method1[method1_cols].round(3))


In [ ]:
# ============================================================
# Cell 16: Compatible Ranking Method 2 — Top-Two minus Bottom-Two Balance
# ============================================================

ranking_method2 = projection_df.copy()

# Weighted 1Y/2Y projected probabilities for each regime.
for name in ORDERED_REGIME_LIST:
    ranking_method2[f"Weighted_{name}"] = (
        0.6 * ranking_method2[f"P_{name}_1Y"] +
        0.4 * ranking_method2[f"P_{name}_2Y"]
    )

# For a 5-regime model:
# Bad side  = bottom two regimes
# Neutral   = middle regime
# Good side = top two regimes
BAD_REGIMES = ORDERED_REGIME_LIST[:2]    # Crisis, Correction
NEUTRAL_REGIMES = ORDERED_REGIME_LIST[2:3]  # Recovery
GOOD_REGIMES = ORDERED_REGIME_LIST[-2:]  # Growth, Boom

print("Bad regimes:", BAD_REGIMES)
print("Neutral regimes:", NEUTRAL_REGIMES)
print("Good regimes:", GOOD_REGIMES)

ranking_method2["Good_Regime_Prob"] = sum(
    ranking_method2[f"Weighted_{name}"] for name in GOOD_REGIMES
)

ranking_method2["Bad_Regime_Prob"] = sum(
    ranking_method2[f"Weighted_{name}"] for name in BAD_REGIMES
)

ranking_method2["Neutral_Regime_Prob"] = sum(
    ranking_method2[f"Weighted_{name}"] for name in NEUTRAL_REGIMES
)

ranking_method2["Method2_Tail_Balance_Score"] = (
    ranking_method2["Good_Regime_Prob"] -
    ranking_method2["Bad_Regime_Prob"]
)

ranking_method2 = ranking_method2.sort_values(
    "Method2_Tail_Balance_Score",
    ascending=False
)

ranking_method2["Rank_Method2"] = np.arange(1, len(ranking_method2) + 1)

method2_cols = [
    "Rank_Method2",
    "Current_Regime",
    "Good_Regime_Prob",
    "Neutral_Regime_Prob",
    "Bad_Regime_Prob",
    "Method2_Tail_Balance_Score",
    "Weighted_Crisis",
    "Weighted_Correction",
    "Weighted_Recovery",
    "Weighted_Growth",
    "Weighted_Boom",
]

print("Method 2: Top-two minus bottom-two regime balance")
display(ranking_method2[method2_cols].round(3))


In [ ]:
# ============================================================
# Cell 17: Combined comparison of compatible ranking methods
# ============================================================

rank_compare = (
    ranking_method1[
        [
            "Rank_Method1",
            "Method1_Expected_Regime_Score",
            "Expected_Regime_Index_0_to_4",
            "Current_Regime",
        ]
    ]
    .join(
        ranking_method2[
            [
                "Rank_Method2",
                "Method2_Tail_Balance_Score",
                "Good_Regime_Prob",
                "Neutral_Regime_Prob",
                "Bad_Regime_Prob",
            ]
        ],
        how="inner"
    )
)

rank_compare["Average_Rank"] = rank_compare[["Rank_Method1", "Rank_Method2"]].mean(axis=1)
rank_compare["Rank_Difference"] = (
    rank_compare["Rank_Method1"] - rank_compare["Rank_Method2"]
).abs()

rank_compare = rank_compare.sort_values(
    ["Average_Rank", "Rank_Difference"],
    ascending=[True, True]
)

rank_compare["Combined_Rank"] = np.arange(1, len(rank_compare) + 1)

comparison_cols = [
    "Combined_Rank",
    "Current_Regime",
    "Rank_Method1",
    "Rank_Method2",
    "Average_Rank",
    "Rank_Difference",
    "Method1_Expected_Regime_Score",
    "Expected_Regime_Index_0_to_4",
    "Method2_Tail_Balance_Score",
    "Good_Regime_Prob",
    "Neutral_Regime_Prob",
    "Bad_Regime_Prob",
]

print("Combined compatible ranking comparison")
display(rank_compare[comparison_cols].round(3))


In [ ]:
# ============================================================
# Cell 18: Transition confidence check
# ============================================================

def transition_counts(regime_seq, n_states=N_REGIMES):
    M = np.zeros((n_states, n_states), dtype=int)
    for t in range(len(regime_seq) - 1):
        M[regime_seq[t], regime_seq[t + 1]] += 1
    return M

confidence_rows = []

for market, grp in panel_scaled.groupby("Market", sort=False):
    grp = grp.sort_values("Date")
    seq = grp["regime"].astype(int).values
    current_state = seq[-1]
    C = transition_counts(seq, N_REGIMES)

    confidence_rows.append({
        "Market": market,
        "Current_Regime": REGIME_NAMES[current_state],
        "Transitions_From_Current_Regime": int(C[current_state].sum()),
        "Visited_Regimes": int(pd.Series(seq).nunique()),
        "Low_Confidence_Projection": bool(C[current_state].sum() < 30),
    })

transition_confidence = pd.DataFrame(confidence_rows).set_index("Market")
display(transition_confidence)


In [ ]:
# ============================================================
# Cell 19: Random-seed robustness for compatible Method 1 and Method 2
# ============================================================

SEEDS = [42, 7, 123, 2024, 99]


def fit_rank_for_seed_5regime(seed):
    model = GaussianHMM(
        n_components=N_REGIMES,
        covariance_type="full",
        n_iter=1000,
        tol=1e-4,
        random_state=seed,
        verbose=False,
        min_covar=1e-4,
    )

    model.fit(X, lengths=lengths)

    tmp = panel_scaled[["Date", "Market", "ret_1d_raw"]].copy()
    tmp["regime_raw"] = model.predict(X, lengths=lengths)

    # Re-order raw states from weakest to strongest by realized daily return.
    order = (
        tmp.groupby("regime_raw")["ret_1d_raw"]
        .mean()
        .sort_values()
        .index.tolist()
    )

    lmap = {old_raw: ordered for ordered, old_raw in enumerate(order)}
    tmp["regime"] = tmp["regime_raw"].map(lmap).astype(int)

    regime_score = {
        i: i / (N_REGIMES - 1)
        for i in range(N_REGIMES)
    }

    bad_idx = list(range(0, 2))       # bottom two
    good_idx = list(range(N_REGIMES - 2, N_REGIMES))  # top two

    rows = []

    for market, grp in tmp.groupby("Market", sort=False):
        grp = grp.sort_values("Date")
        seq = grp["regime"].values.astype(int)

        T = empirical_transition_matrix(seq, N_REGIMES)

        current_state = seq[-1]
        e = np.zeros(N_REGIMES)
        e[current_state] = 1.0

        d1 = e @ np.linalg.matrix_power(T, TRADING_DAYS_1Y)
        d2 = e @ np.linalg.matrix_power(T, TRADING_DAYS_2Y)

        weighted = 0.6 * d1 + 0.4 * d2

        # Method 1: expected normalized regime score
        method1_score = sum(
            weighted[i] * regime_score[i]
            for i in range(N_REGIMES)
        )

        # Method 2: top-two minus bottom-two balance
        good_prob = weighted[good_idx].sum()
        bad_prob = weighted[bad_idx].sum()
        method2_score = good_prob - bad_prob

        rows.append({
            "Market": market,
            "Method1_Score": method1_score,
            "Method2_Score": method2_score,
        })

    r = pd.DataFrame(rows).set_index("Market")
    r[f"M1_seed_{seed}"] = r["Method1_Score"].rank(ascending=False, method="min")
    r[f"M2_seed_{seed}"] = r["Method2_Score"].rank(ascending=False, method="min")

    return r[[f"M1_seed_{seed}", f"M2_seed_{seed}"]]


seed_rank_table_5 = None

for seed in SEEDS:
    r = fit_rank_for_seed_5regime(seed)
    seed_rank_table_5 = r if seed_rank_table_5 is None else seed_rank_table_5.join(r, how="outer")

m1_cols = [c for c in seed_rank_table_5.columns if c.startswith("M1_seed_")]
m2_cols = [c for c in seed_rank_table_5.columns if c.startswith("M2_seed_")]

seed_rank_table_5["M1_Mean_Rank"] = seed_rank_table_5[m1_cols].mean(axis=1)
seed_rank_table_5["M2_Mean_Rank"] = seed_rank_table_5[m2_cols].mean(axis=1)
seed_rank_table_5["M1_Rank_Std"] = seed_rank_table_5[m1_cols].std(axis=1)
seed_rank_table_5["M2_Rank_Std"] = seed_rank_table_5[m2_cols].std(axis=1)
seed_rank_table_5["Avg_Rank_Std"] = seed_rank_table_5[["M1_Rank_Std", "M2_Rank_Std"]].mean(axis=1)

seed_rank_table_5 = seed_rank_table_5.sort_values(
    ["Avg_Rank_Std", "M1_Mean_Rank"],
    ascending=[True, True]
)

print("5-regime seed robustness for compatible ranking methods")
display(seed_rank_table_5.round(2))


In [ ]:
# ============================================================
# Cell 20: Single-panel chart — Method 1 ranking
# ============================================================

plot_df = ranking_method1.sort_values("Method1_Expected_Regime_Score")

plt.figure(figsize=(10, 6))
plt.barh(plot_df.index, plot_df["Method1_Expected_Regime_Score"])
plt.title("5-Regime HMM — Method 1: Expected Normalized Regime Score")
plt.xlabel("Expected Regime Score, 0 = weakest and 1 = strongest")
plt.ylabel("Market")
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# Cell 21: Single-panel chart — Method 2 ranking
# ============================================================

plot_df = ranking_method2.sort_values("Method2_Tail_Balance_Score")

plt.figure(figsize=(10, 6))
plt.barh(plot_df.index, plot_df["Method2_Tail_Balance_Score"])
plt.title("5-Regime HMM — Method 2: Top-Two minus Bottom-Two Balance")
plt.xlabel("Good regime probability - Bad regime probability")
plt.ylabel("Market")
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# Cell 22: Save all important outputs
# ============================================================

suffix = "price_vix_5regime_compatible"

ranking_method1.to_csv(f"market_ranking_method1_{suffix}.csv")
ranking_method2.to_csv(f"market_ranking_method2_{suffix}.csv")
rank_compare.to_csv(f"market_ranking_compare_{suffix}.csv")
projection_df.to_csv(f"market_regime_projection_{suffix}.csv")
current_regime_prob_table.to_csv(f"current_regime_probabilities_{suffix}.csv")
regime_validation.to_csv(f"regime_validation_{suffix}.csv")
regime_share_by_market.to_csv(f"regime_share_by_market_{suffix}.csv")
transition_confidence.to_csv(f"transition_confidence_{suffix}.csv")
seed_rank_table_5.to_csv(f"seed_rank_table_{suffix}.csv")

with pd.ExcelWriter(f"panel_hmm_{suffix}_outputs.xlsx", engine="openpyxl") as writer:
    availability.to_excel(writer, sheet_name="Availability", index=False)
    common_diagnostics.to_excel(writer, sheet_name="Common_Diagnostics", index=False)
    regime_validation.to_excel(writer, sheet_name="Regime_Validation")
    regime_share_by_market.to_excel(writer, sheet_name="Regime_Share")
    current_regime_prob_table.to_excel(writer, sheet_name="Current_Regime_Prob")
    projection_df.to_excel(writer, sheet_name="Projection")
    ranking_method1.to_excel(writer, sheet_name="Ranking_Method1")
    ranking_method2.to_excel(writer, sheet_name="Ranking_Method2")
    rank_compare.to_excel(writer, sheet_name="Ranking_Compare")
    transition_confidence.to_excel(writer, sheet_name="Transition_Confidence")
    seed_rank_table_5.to_excel(writer, sheet_name="Seed_Robustness")

print("Saved outputs:")
print(f"- market_ranking_method1_{suffix}.csv")
print(f"- market_ranking_method2_{suffix}.csv")
print(f"- market_ranking_compare_{suffix}.csv")
print(f"- panel_hmm_{suffix}_outputs.xlsx")


## Interpretation reminder

This 5-regime notebook ranks markets by **regime-transition structure**, not by guaranteed return forecasts.

The ranking methods are now compatible with different numbers of regimes because they use relative regime position rather than hard-coded 4-regime labels.

- **Method 1** uses the full projected 5-regime distribution and maps each regime to a normalized score from 0 to 1.
- **Method 2** uses a comparable tail-balance framework: top-two regimes minus bottom-two regimes. The middle Recovery state is neutral.

Therefore, the 5-regime ranking should be interpreted as:

```text
Which markets are projected to have more probability mass in stronger regimes and less probability mass in weaker regimes?
```

This makes the 5-regime model easier to compare with the earlier 4-regime version.
